# 01 · Ingestion & EDA (V1 milestone M0)

Stages S0–S1 of `version_1.md`:
- read every TSV safely: strings only, no NA detection, no quoting
- integrity checks
- check assumptions **A1–A7**
- check the all-empty baseline in the scoring harness

The file sizes are about 2.2M S1 rows in train and 10M S2/S3 rows, so everything runs on **polars**.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # run the notebook from the `version 1` folder
import numpy as np, polars as pl
import er
from er import log
pl.Config.set_fmt_str_lengths(90); pl.Config.set_tbl_width_chars(220); pl.Config.set_tbl_rows(30)

polars.config.Config

## S0 · Load + integrity checks

In [2]:
t = time.time()
tr1, tr23, gt = er.load_split("train")
te1, te23, _ = er.load_split("test")
log(f"loaded in {time.time()-t:.1f}s")

def n_lines(p):
    with open(p, "rb") as f:
        return sum(1 for _ in f) - 1

for split in ["train", "test"]:
    for s in [1, 2, 3]:
        p = er.DATA / split / f"{split}_source{s}.tsv"
        df = er.read_tsv(p)
        assert df.height == n_lines(p), p                         # no rows lost to quoting
        assert df["entity_id"].str.starts_with(f"S{s}-").all(), p  # prefix matches file
        assert df["entity_id"].n_unique() == df.height and (df["entity_id"] != "").all(), p
        print(f"{split} S{s}: {df.height:>10,} rows  OK")
assert gt.height == tr1.height and set(gt["source1_entity_id"]) == set(tr1["entity_id"])
print("ground truth covers every train S1  OK")

[17:49:13] loaded in 4.3s


train S1:  2,206,821 rows  OK


train S2:  5,034,616 rows  OK


train S3:  5,285,603 rows  OK


test S1:  1,732,544 rows  OK


test S2:  4,887,273 rows  OK


test S3:  5,082,316 rows  OK


ground truth covers every train S1  OK


## Record counts per source × country, train vs test (A7: pool density)

In [3]:
def counts(split, s1, s23):
    return (pl.concat([s1.with_columns(src=pl.lit("S1")),
                       s23.with_columns(src=pl.col("entity_id").str.slice(0, 2))])
              .group_by("country", "src").len().with_columns(split=pl.lit(split)))

c = pl.concat([counts("train", tr1, tr23), counts("test", te1, te23)])
tab = c.pivot(on="src", index=["split", "country"], values="len").sort("split", "country")
print(tab.with_columns(s23_per_s1=((pl.col("S2") + pl.col("S3")) / pl.col("S1")).round(2)))

shape: (5, 6)
┌───────┬─────────┬─────────┬─────────┬─────────┬────────────┐
│ split ┆ country ┆ S2      ┆ S1      ┆ S3      ┆ s23_per_s1 │
│ ---   ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---        │
│ str   ┆ str     ┆ u32     ┆ u32     ┆ u32     ┆ f64        │
╞═══════╪═════════╪═════════╪═════════╪═════════╪════════════╡
│ test  ┆ France  ┆ 703378  ┆ 259452  ┆ 731615  ┆ 5.53       │
│ test  ┆ India   ┆ 2312565 ┆ 809986  ┆ 2405000 ┆ 5.82       │
│ test  ┆ US      ┆ 1871330 ┆ 663106  ┆ 1945701 ┆ 5.76       │
│ train ┆ India   ┆ 2017799 ┆ 883188  ┆ 2115547 ┆ 4.68       │
│ train ┆ US      ┆ 3016817 ┆ 1323633 ┆ 3170056 ┆ 4.67       │
└───────┴─────────┴─────────┴─────────┴─────────┴────────────┘


## Ground-truth structure (singletons, matches per S1, A1, A2, A4)

In [4]:
pairs = er.gt_pairs(gt)
k = gt.select("source1_entity_id", k=pl.col("matched_entity_ids").str.split(",")
              .list.eval(pl.element().filter(pl.element() != "")).list.len())
k = k.join(tr1.select(pl.col("entity_id").alias("source1_entity_id"), "country"), on="source1_entity_id")
print(f"match pairs: {pairs.height:,}   mean matches/S1: {k['k'].mean():.2f}")
print("singleton rate by country:")
print(k.group_by("country").agg(singleton_rate=(pl.col("k") == 0).mean(), n=pl.len(), mean_k=pl.col("k").mean()))
print("distribution of matches per S1:")
print(k["k"].value_counts().sort("k").with_columns(share=(pl.col("count") / k.height).round(4)).transpose(include_header=True))
print(f"share of matches from S2: {pairs['s23_id'].str.starts_with('S2').mean():.3f}")

a1 = pairs.group_by("s23_id").len().filter(pl.col("len") > 1).height
print(f"\nA1  S2/S3 IDs under more than one S1: {a1}  ->", "CONFIRMED (exclusivity is safe)" if a1 == 0 else "VIOLATED")

matched = set(pairs["s23_id"])
for src in ["S2", "S3"]:
    ids = tr23.filter(pl.col("entity_id").str.starts_with(src))["entity_id"]
    print(f"A2  {src} records matching no S1: {1 - ids.is_in(matched).mean():.3f}")

j = (pairs.join(tr1.select(pl.col("entity_id").alias("s1_id"), "country"), on="s1_id")
          .join(tr23.select(pl.col("entity_id").alias("s23_id"), pl.col("country").alias("country_r")), on="s23_id"))
print(f"A4  matched pairs sharing the country label: {(j['country'] == j['country_r']).mean():.4f}")

match pairs: 7,638,365   mean matches/S1: 3.46
singleton rate by country:
shape: (2, 4)
┌─────────┬────────────────┬─────────┬──────────┐
│ country ┆ singleton_rate ┆ n       ┆ mean_k   │
│ ---     ┆ ---            ┆ ---     ┆ ---      │
│ str     ┆ f64            ┆ u32     ┆ f64      │
╞═════════╪════════════════╪═════════╪══════════╡
│ India   ┆ 0.055878       ┆ 883188  ┆ 3.464543 │
│ US      ┆ 0.055828       ┆ 1323633 ┆ 3.459057 │
└─────────┴────────────────┴─────────┴──────────┘
distribution of matches per S1:
shape: (3, 13)
┌────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬───────────┬───────────┐
│ column ┆ column_0 ┆ column_1 ┆ column_2 ┆ … ┆ column_8 ┆ column_9 ┆ column_10 ┆ column_11 │
│ ---    ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ ---       ┆ ---       │
│ str    ┆ f64      ┆ f64      ┆ f64      ┆   ┆ f64      ┆ f64      ┆ f64       ┆ f64       │
╞════════╪══════════╪══════════╪══════════╪═══╪══════════╪══════════╪═══════════╪════════

share of matches from S2: 0.484



A1  S2/S3 IDs under more than one S1: 0  -> CONFIRMED (exclusivity is safe)


A2  S2 records matching no S1: 0.266


A2  S3 records matching no S1: 0.254


A4  matched pairs sharing the country label: 1.0000


## A3 · Duplicates inside S2/S3 (several records of the same entity from one source)

In [5]:
per = pairs.with_columns(src=pl.col("s23_id").str.slice(0, 2)).group_by("s1_id", "src").len()
print(per.group_by("src").agg(mean_per_s1=pl.col("len").mean(), share_s1_with_2plus=(pl.col("len") >= 2).mean()))

shape: (2, 3)
┌─────┬─────────────┬─────────────────────┐
│ src ┆ mean_per_s1 ┆ share_s1_with_2plus │
│ --- ┆ ---         ┆ ---                 │
│ str ┆ f64         ┆ f64                 │
╞═════╪═════════════╪═════════════════════╡
│ S2  ┆ 1.924686    ┆ 0.588808            │
│ S3  ┆ 2.032803    ┆ 0.630817            │
└─────┴─────────────┴─────────────────────┘


## Missingness and junk values per source

In [6]:
junk = ["", "None", "none", "NULL", "<NULL>", "N/A", "nan"]
def miss(df, name):
    return df.group_by("country").agg(
        pl.lit(name).alias("src"),
        addr_missing=pl.col("business_address").str.strip_chars().is_in(junk).mean(),
        addr_has_digit=pl.col("business_address").str.contains(r"\d").mean(),
        name_non_ascii=pl.col("business_name").str.contains(r"[^\x00-\x7F]").mean(),
        name_has_url=pl.col("business_name").str.contains(r"\.(com|in|net|org|fr)\b").mean())
print(pl.concat([miss(tr1, "S1"), miss(tr23.filter(pl.col("entity_id").str.starts_with("S2")), "S2"),
                 miss(tr23.filter(pl.col("entity_id").str.starts_with("S3")), "S3"),
                 miss(te1, "test S1"), miss(te23, "test S2/S3")]).sort("country", "src"))

shape: (12, 6)
┌─────────┬────────────┬──────────────┬────────────────┬────────────────┬──────────────┐
│ country ┆ src        ┆ addr_missing ┆ addr_has_digit ┆ name_non_ascii ┆ name_has_url │
│ ---     ┆ ---        ┆ ---          ┆ ---            ┆ ---            ┆ ---          │
│ str     ┆ str        ┆ f64          ┆ f64            ┆ f64            ┆ f64          │
╞═════════╪════════════╪══════════════╪════════════════╪════════════════╪══════════════╡
│ France  ┆ test S1    ┆ 0.0          ┆ 0.995803       ┆ 0.157212       ┆ 0.0          │
│ France  ┆ test S2/S3 ┆ 0.03002      ┆ 0.932689       ┆ 0.242372       ┆ 0.026922     │
│ India   ┆ S1         ┆ 0.0          ┆ 0.912775       ┆ 0.0            ┆ 0.0          │
│ India   ┆ S2         ┆ 0.028668     ┆ 0.913165       ┆ 0.278738       ┆ 0.027146     │
│ India   ┆ S3         ┆ 0.0307       ┆ 0.900555       ┆ 0.184903       ┆ 0.029031     │
│ India   ┆ test S1    ┆ 0.0          ┆ 0.912655       ┆ 0.0            ┆ 0.0          │
│ Indi

## Sample matched pairs (what the noise looks like)

In [7]:
smp = (pairs.sample(25, seed=1)
       .join(tr1.select(pl.col("entity_id").alias("s1_id"), "business_name", "business_address"), on="s1_id")
       .join(tr23.select(pl.col("entity_id").alias("s23_id"), pl.col("business_name").alias("name_r"),
                         pl.col("business_address").alias("addr_r")), on="s23_id"))
for r in smp.iter_rows(named=True):
    print(f"{r['business_name']}  |  {r['business_address']}\n    -> {r['name_r']}  |  {r['addr_r']}")

Federal Wellness Advisors Harwood  |  234 Buffalo Run, Harwood, TX
    -> federalwellnessadvisors.com  |  234 Buffalo Run, HARWOOD, TX
Siva House (India) Pvt Ltd  |  Ahmedabad, Sahjanand Estate, Village: Sarkhej, Ahmedabad, Godown - 08, Ahmedabad, Gujarat
    -> Sri Siva Hóuse (India) Pvt Ltd  |  GODOWN - ##08, SAHJANAND ESTATE, VILLAGE: SARKHEJ, AHMEDABAD, AHMEDABAD, Gujarat
Systech Products  |  Kolkata, West Bengal, Block Bc No 96 Salt Lake City, North 24 Parganas
    -> Systech Products LLP  |  BLOCK BC NO 96 SALT LAKE CITY, KOLKATA, West Bengal
Frontier Mobile LLC  |  4218 Marten Court, Unit APT 137, Indianapolis, IN
    -> Frontier Móbile LLC  |  04218 MARTEN CT, INDIANAPOLIS, IN
Vance and Nguyen Enclave  |  6350 Christine Dr, Marion, IA
    -> Vance and Ngúyen Enclave  |  6350 CHRISTINE DR, MARION, IA
Maid Yoga  |  571 Judi Street, Casa Grande, AZ
    -> MAID YOGA  |  571-575 JUDI ST, PO BOX 2922, CASA GRANDE, AZ
Rishta & Brothers Private Limited  |  3Rd Floor, Shop No. 02, Satya

## France (test only): what the unseen country looks like

In [8]:
fr = te1.filter(pl.col("country").str.to_lowercase().str.strip_chars() == "france")
print(f"France S1 in test: {fr.height:,} ({fr.height / te1.height:.1%} of test S1)")
print(fr.sample(10, seed=0).select("business_name", "business_address"))
print(te23.filter(pl.col("country") == "France").sample(10, seed=0).select("entity_id", "business_name", "business_address"))

France S1 in test: 259,452 (15.0% of test S1)
shape: (10, 2)
┌───────────────────────────────────┬──────────────────────────────────────────────────────────┐
│ business_name                     ┆ business_address                                         │
│ ---                               ┆ ---                                                      │
│ str                               ┆ str                                                      │
╞═══════════════════════════════════╪══════════════════════════════════════════════════════════╡
│ Club de Gaia                      ┆ 178 Boulevard Lafayette, Calais, Hauts-de-France         │
│ Meltdown Maternelle (France) SARL ┆ Hauts-de-France, Calais, 567 Rue de Puébla               │
│ École maternelle Saint Pachamama  ┆ 295 Route de Sainte Luce, Nantes, Pays de la Loire       │
│ Sociale Ecole SASU                ┆ 70 Rue Dubourdieu, Bordeaux, Nouvelle-Aquitaine          │
│ Pessac Store EURL                 ┆ 17 Rue de la Fon de Madran, 

## Most frequent last name-tokens per country (seeds for legal-form discovery, V1 §6.3)

In [9]:
def last_tokens(df):
    return (df.with_columns(t=pl.col("business_name").map_elements(lambda s: (er.norm(s).split() or [""])[-1], return_dtype=pl.String))
              .group_by("country", "t").len().sort("len", descending=True).group_by("country").head(15))
lt = last_tokens(pl.concat([tr1.sample(200_000, seed=0), te1.filter(pl.col("country") == "France")]))
for c_, g in lt.group_by("country"):
    print(c_[0], ":", ", ".join(f"{t}({n})" for t, n in g.select("t", "len").iter_rows()))

US : llc(32334), inc(21610), c(3993), corp(3042), group(2421), pc(2306), pllc(1796), center(1721), associate(1608), lp(1541), partners(1327), co(1240), clinic(1231), care(1208), holdings(513)
India : ltd(60330), llp(3531), co(2176), corp(1220), trust(614), group(503), clinic(428), associate(418), society(416), partners(414), care(364), school(320), foundation(287), center(280), sangh(242)
France : sarl(73480), sas(52276), eurl(16979), sa(12753), sasu(10721), sci(8360), ei(4182), club(2619), jean(1713), france(1577), ecole(1494), amicale(1446), comite(1399), sainte(1304), sportive(1155)


## Scoring harness check (M0 exit criterion)
The all-empty prediction must score exactly the singleton rate.

In [10]:
ids = tr1["entity_id"].to_list()
base = er.macro_f05(pairs.head(0), pairs, ids)
sr = (k["k"] == 0).mean()
print(f"all-empty macro F0.5 = {base:.5f}   singleton rate = {sr:.5f}")
assert abs(base - sr) < 1e-9

all-empty macro F0.5 = 0.05585   singleton rate = 0.05585


## Decision log
- **A1 confirmed.** No S2/S3 ID sits under two S1s, so each S2/S3 record keeps only its best S1 (exclusivity).
- **A2 confirmed.** About a quarter of S2/S3 records match no S1. They act as distractors, so rejecting them matters.
- **A3 confirmed.** S1 entities usually have several records from the same source, so returning more than one match per S1 is the norm.
- **A4 confirmed.** Matched pairs always share the country label, so blocking runs within each country. Country labels stay an open set, so France blocks against the French pool.
- **Singletons are about 5.6%.** The all-empty baseline is about 0.056, and the average S1 has about 3.5 matches. **Recall across the full list matters, not just the top-1 match.**
- **Noise includes:**
  - native-script names (Devanagari, Gujarati, Malayalam) and state names
  - name tokens replaced by a URL or a random word
  - injected accents
  - `None` / `<NULL>` addresses
  - `##` / `#` prefixes and reordered address components

  Because of this, **blocking uses name keys and address keys together**.
- **A7: the test pool is denser.** Test has 5.5–5.8 S2/S3 records per S1; train has 4.7. That means more distractors per S1 on test, so precision is at risk. The training sample keeps the train density, so we watch the predicted-empty rate and mean matches per country in the shift monitor (notebook 02, §11).
- **France is 15% of test S1.** Its legal forms (sarl, sas, eurl, sasu, sci) show up in the frequency-based last-token scan without any hard-coding.